In [2]:
%%writefile requirements.txt
numpy==2.3.3
pandas==2.3.3
pmdarima==2.1.1
polars==1.34.0
polars-runtime-32==1.34.0
scikit-learn==1.7.2
statsmodels==0.14.5
xgboost==3.1.1
yfinance==0.2.66

Overwriting requirements.txt


In [3]:
import os

def make_directory(path: str) -> None:
    try:
        os.makedirs(path)
        print(f"created directory '{path}'!")
    except FileExistsError:
        print(f"directory '{path}' already exists!")

make_directory("src")
make_directory("workflows")
make_directory("src/xgboost_model")
make_directory("src/sarimax_model")
make_directory("src/prophet_model")

directory 'src' already exists!
directory 'workflows' already exists!
directory 'src/xgboost_model' already exists!
directory 'src/sarimax_model' already exists!
directory 'src/prophet_model' already exists!


# loading the raw data from `yfinance`

this serves as the base function for loading all data

In [2]:
%%writefile src/load_data.py
"""
function written to easily load and process stock data from `yfinance`.
"""
import pandas as pd
import yfinance as yf
import polars as pl

def load_stocks(
    stocks: str | list[str], start: str, end: str, use_polars: bool = True
) -> pl.DataFrame | pd.DataFrame:
    """load stock data from `yfinance`.

    stocks are loaded singularly, from start to end date. option to return a 
    polars dataframe or a pandas dataframe.

    Args:
        stocks (list): single-item list of stock tockers.
        start (str): first historical date.
        end (str): last historical date (up to today).
        use_polars (bool, optional): whether to return a polars dataframe or a
            pandas dataframe. defaults to true.

    Raises:
        ValueError: raised if users enter more than 1 ticker

    Returns:
        DataFrame: polars or pandas dataframe, depending on the value of
        `use_polars`.
    """
    if isinstance(stocks, str):
        stocks = [stocks]
        
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")
        
    df = yf.download(stocks, start, end)
    df.index = pd.to_datetime(df.index)
    df.columns = (
        pd.MultiIndex.from_tuples(df.columns) 
        if not isinstance(df.columns, pd.MultiIndex) else df.columns
    )
    df.columns = df.columns.set_names(["Field", "Ticker"])
    df.index.name = "Date"
    df = df.apply(pd.to_numeric, errors="coerce")

    df_out = (
        df.swaplevel("Field", "Ticker", axis=1)
        .sort_index(axis=1)
        .stack("Ticker", future_stack=True)
        .reset_index()
    )

    df_out = df_out.rename(columns=str.lower)

    return pl.from_pandas(df_out) if use_polars else df_out

Overwriting src/load_data.py


# data transformations and feature engineering

## xgboost

In [3]:
%%writefile src/xgboost_model/xgboost_etl.py
"""
set of functions to process `yfinance` data for the XGBoost model.

adds lagged features and time indicators to build the full feature space for
each model.
"""
from __future__ import annotations 

import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple, Literal

from src.load_data import load_stocks


def prep_columns(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """prep the columns pulled from `yfinance` into a clean dataframe

    adds a 'move' column to indicate overall daily change; time-lapse columns
    lagged over 1, 7, 30 days; and rolling mean and SD values over 7 days.

    Args:
        df (pl.DataFrame): raw `yfinance` dataframe.
        col (str): which column (choose between 'close', 'open') to compute the
            lag features for.

    Returns:
        pl.DataFrame: full dataframe with lagged features of chosen column.
    """
    if col == "move":
        df = df.with_columns(
            (pl.col("close") - pl.col("open")).alias(col)
        )

    df_out =  (
        df.select(
            [
                "date",
                "ticker",
                "volume",
                col
            ]
        )
        .sort([pl.col("ticker"), pl.col("date")], descending=False)
        .with_columns(
            pl.col(col).shift(1).over("ticker").alias(f"prev1_{col}"),
            pl.col(col).shift(7).over("ticker").alias(f"prev7_{col}"),
            pl.col(col).shift(30).over("ticker").alias(f"prev30_{col}"),
        )
        .with_columns(
            pl.col(col)
            .rolling_mean(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_mean_7")
        )
        .with_columns(
            pl.col(col)
            .rolling_std(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_std_7")
        )
    )

    return df_out


def prep_data_frame(df: pl.DataFrame) -> pl.DataFrame:
    """prepare lag columns for all of 'open', 'close', and 'move'.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.

    Returns:
        pl.DataFrame: processed stock data with lag columns for all price
            indicators.
    """
    markers = ["open", "close", "move"]
    df_out = None

    for marker in markers:
        df_prep = prep_columns(df, marker)
        if df_out is None:
            df_out = df_prep
        else:
            df_out = df_prep.join(
                df_out, on=["date", "ticker", "volume"], how="inner"
            )
    
    return (
        df_out.with_columns(
            pl.col("date").dt.weekday().alias("dow")
        )
        .with_columns(
            pl.col("date").dt.month().alias("month")
        )
        .with_columns(
            pl.when(pl.col("dow").is_in([0, 4]))
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias("mon_or_fri")
        )
    )

LabelMode = Literal["log_return", "simple_return"]

def build_dataset(
    df: pl.DataFrame, 
    label_col: str = "close", 
    horizon: int = 21, 
    label_mode: LabelMode = "log_return"
) -> pl.DataFrame:
    """wrapper for `prep_data_frame`.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.
        label (str, optional): which of 'close' or 'move' to process. defaults
            to "close".
        horizon (int, optional): length of forecast horizon. must be >= 1.
        label_mode (Literal): 

    Raises:
        ValueError: only accepts `horizon` values >= 1.
        ValueError: only accepts 'close' or 'move' or 'open'.
        ValueError: only accepts 'log_return' or 'simple_return'.

    Returns:
        pl.DataFrame: fully processed `yfinance` data with nulls removed.
    """
    if label_col not in {"close", "open", "move"}:
        raise ValueError("`label_col` must be one of ['close', 'open', 'move']")
    
    df_feat = prep_data_frame(df)

    if horizon < 1:
        raise ValueError("`horizon` must be >= 1")

    if label_col == "move":
        base = pl.col("move")
    else:
        base = pl.col(label_col)
    
    future = base.shift(-horizon)

    if label_mode == "log_return":
        df_feat = df_feat.with_columns(
            (future.log() - base.log()).alias("label")
        )
    elif label_mode == "simple_return":
        df_feat = df_feat.with_columns(((future / base) - 1.0).alias("label"))
    else:
        raise ValueError(
            "`labl_mode` must be one of ['log_return', 'simple_return']"
        )

    return df_feat.drop_nulls()

Overwriting src/xgboost_model/xgboost_etl.py


## sarimax

In [6]:
%%writefile src/sarimax_model/sarimax_etl.py
"""
set of functions to process `yfinance` data for the SARIMAX model.

pulls code from xgboost model (`build_dataset`) and uses it to add indexes to 
the dataframe. 
"""
import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple

from src.load_data import load_stocks
from src.xgboost_model.xgboost_etl import build_dataset


def build_df_with_indices(
    df: pl.DataFrame, label: str, start: str, end: str
) -> pl.DataFrame:
    """process raw dataframe and add indices.

    this is originally intended to bolster the SARIMAX model by adding broad
    exogenous variables to the features space.

    Args:
        df (pl.DataFrame): raw `yfiance` stock dataframe.
        label (str): 'close' or 'move' price indicator.
        start (str): start date for index pulling from `yfinance`.
        end (str): end date for index pulling from `yfinance`.

    Returns:
        pl.DataFrame: features data with lagged columns and added index values.
    """
    indices_list = ["SPY", "QQQ", "IWM", "VXX", "UUP", "HYG", "LQD"]
    df_idx_out = None

    for idx in indices_list:
        idx_cl = idx.replace("^", "")

        df_idx = load_stocks([idx], start, end).select(
            pl.col("date"),
            pl.col("close").alias(f"{idx_cl}_close"),
            pl.col("volume").alias(f"{idx_cl}_volume")
        )

        if df_idx_out is None:
            df_idx_out = df_idx
        else:
            df_idx_out = df_idx_out.join(df_idx, on=["date"], how="inner")
    
    df_ticker = build_dataset(df, label)

    df_out = df_ticker.join(df_idx_out, on=["date"], how="inner")
    
    return df_out

Overwriting src/sarimax_model/sarimax_etl.py


## prophet

In [7]:
%%writefile src/prophet_model/prophet_etl.py
"""
set of functions to process `yfinance` data for the Prophet model.

focuses on pulling indexes and computing returns and volatility to add these as
regressors in Prophet.
"""
import yfinance as yf 
import polars as pl 
import pandas as pd
from typing import Tuple

from src.load_data import load_stocks 


class GetSectorETF:
    """simple class to infer sector ETFs"""
    def __init__(
        self,
        indexes: list,
        label: str,
        target_ticker: str
    ):
        self.SECTOR_TO_ETF = {
            "Technology": "XLK",
            "Communication Services": "XLC",
            "Financial Services": "XLF",
            "Energy": "XLE",
            "Consumer Cyclical": "XLY",
            "Consumer Defensive": "XLP",
            "Industrials": "XLI",
            "Healthcare": "XLV",
            "Real Estate": "XLRE",
            "Utilities": "XLU",
        }

        self.indexes = indexes
        self.target_ticker = target_ticker

        if label not in ["open", "close", "move"]:
            raise ValueError("label must be one of ['open', 'close', 'move]")
        else:
            self.label = label

    def extract_stock_info(self, stock_df: pl.DataFrame) -> Tuple[str, str, str]:
        """extract the ticker, earliest, and latest data from `stock_df`.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            Tuple[str, str, str]: ticker, start_date, end_date values
        """
        tickers = [stock_df.select("ticker").unique().item()]

        date_df = (
            stock_df
            .select("date")
            .unique()
            .sort(by="date", descending=True)
        )

        min_date = date_df.select("date").tail(1).item().strftime("%Y-%m-%d")
        max_date = date_df.select("date").head(1).item().strftime("%Y-%m-%d")

        return tickers, min_date, max_date

    def infer_sector_etfs(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """using tickers, extracts the close values of sector ETFs.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: dataframe of dates, `label` values for relevant ETFs 
                for the stock's ticker, as well as `label` and values for the
                passed-in indexes.
        """
        tickers, start_date, end_date = self.extract_stock_info(stock_df)

        etfs = set()

        for t in tickers:
            info = yf.Ticker(t).info
            sector = info.get("sector")
            if sector in self.SECTOR_TO_ETF:
                etfs.add(self.SECTOR_TO_ETF[sector])
        
        ticker_list = list(etfs)
        ticker_list += self.indexes
        df_out = None

        for ticker in ticker_list:
            df_temp = load_stocks([ticker], start_date, end_date).select(
                pl.col("date"),
                pl.col(self.label).alias(f"{ticker}")
            )

            if df_out is None:
                df_out = df_temp 
            else:
                df_out = df_out.join(df_temp, on=["date"], how="inner")
        
        return df_out
    
    def build_etf_df(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """combine ETF values with stock dataframe

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: combined dataframe of `stock_df` with ETF values.
        """
        df_etf = self.infer_sector_etfs(stock_df)

        return (
            stock_df.select(
                pl.col("date"), pl.col(self.label).alias(self.target_ticker)
            )
            .join(
                df_etf, on=["date"], how="inner"
            )
            .sort("date")
        )

def compute_returns(df: pl.DataFrame) -> pl.DataFrame:
    """compute the daily return rate.

    Args:
        df (pl.DataFrame): stock dataframe loaded from `yfinance` and processed
            through `GetSectorETF`.

    Returns:
        pl.DataFrame: polars dataframe with daily returns for a target stock and
            desired indexes.
    """
    tickers = [c for c in df.columns if c != "date"]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker).pct_change().alias(f"{ticker}_return")
        )
    
    return df

def compute_return_volatility(df: pl.DataFrame, window: int) -> pd.DataFrame:
    """compute the rolling volatility (SD) of target stock and indexes.

    returns a pandas dataframe for use in prepping data for Prophet.

    Args:
        df (pl.DataFrame): polars dataframe with target stock and index prices
            and returns.
        window (int): rolling window (in days) to compute volatility.

    Returns:
        pd.DataFrame: pandas dataframe with prices, returns, and return 
            volatility.
    """
    tickers = [c for c in df.columns if c.endswith("_return")]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker)
            .rolling_std(window_size=window, min_samples=1)
            .alias(f"{ticker}_rolling_std_{window}")
        )
    
    return df.to_pandas()

def compute_rsi(df: pd.DataFrame, ticker: str, period: int) -> pd.Series:
    """add a column for RSI over a period window.

    Args:
        df (pd.DataFrame): pandas dataframe with target stock PRICES.
        ticker (str): stock to compute RSI for.
        period (int): window (days).

    Returns:
        pd.Series: pandas series to add to `df` as a column containing RSI
            values.
    """
    prices = pd.to_numeric(df[ticker], errors="coerce")
    delta = prices.diff()

    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

def compute_price_volatility(
    df: pd.DataFrame, ticker: str, period: int
) -> pd.Series:
    """add price volatility columns.

    Args:
        df (pd.DataFrame): pandas dataframe with price data.
        ticker (str): stock to compute volatility for.
        period (int): window (days).

    Returns:
        pd.Series: column indicating lagged prices.
    """
    return df[ticker].shift(period)

def build_prophet_df(
    stocks: list,
    start: str,
    end: str,
    label: str,
    indexes: list
) -> pd.DataFrame:
    """run through suite of functions to build the final dataset for Prophet.

    Args:
        stocks (list): _description_
        start (str): _description_
        end (str): _description_
        label (str): _description_
        indexes (list): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tkr = stocks[0]

    df_raw = load_stocks(stocks, start, end)
    etfs = GetSectorETF(indexes=indexes, label=label, target_ticker=tkr)
    df_etf = etfs.build_etf_df(df_raw)

    df_ret = compute_returns(df_etf)
    df_vol = compute_return_volatility(df_ret, 10)

    df_vol["rsi_7"] = compute_rsi(df_vol, tkr, 7)
    df_vol["rsi_14"] = compute_rsi(df_vol, tkr, 14)
    df_vol["rsi_21"] = compute_rsi(df_vol, tkr, 21)
    df_vol["prev7_close"] = compute_price_volatility(df_vol, tkr, 1)
    df_vol["prev14_close"] = compute_price_volatility(df_vol, tkr, 7)
    df_vol["prev30_close"] = compute_price_volatility(df_vol, tkr, 30)

    return (
        df_vol.rename(columns={"date": "ds", f"{tkr}": "y"})
        .dropna()
    )

Overwriting src/prophet_model/prophet_etl.py


# prep the data for modeling

mostly just used for train-test splits but different models call for different  
data parsing (even if only slightly)

In [4]:
%%writefile src/model_preprocess.py
""" 
handle the creation of train-test splits for each model. not all are the same.
split for xgboost is traditional (X,y train/test tables), but for forecasting
models the split is a train/eval split without a test dataframe.

each split is done based on a cutoff date to only allow training on past data 
and testing/eval on future data.
"""
from datetime import datetime
import polars as pl
import pandas as pd
from typing import Tuple

def train_test_split_cutoff(
    df: pl.DataFrame, cutoff: datetime, label_col: str
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """do a train-test split based on a cutoff value.

    training data is all data prior to the cutoff, testing data is all data on
    or after the cutoff.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label_col (str): label column to indicate which is 'y'.

    Returns:
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]: gives the
            "traditional" X_train, X_test, y_train, y_test output (akin to
            sklearn).
    """
    train = df.filter(pl.col("date") < cutoff)
    test = df.filter(pl.col("date") >= cutoff)

    X_train, X_test = (
        train.drop(label_col, "ticker", "date"),
        test.drop(label_col, "ticker", "date")
    )
    y_train, y_test = train[label_col], test[label_col]

    return X_train, X_test, y_train, y_test

def split_ar_on_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str, exog_feats: list
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """split autoregressive data on a cutoff value.

    this doesn't return unique tables for X and y and is specifically designed
    for SARIMAX. can also be used for training Prophet. a pandas dataframe is
    returned (not polars) for use in the forecasting models.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is the endogenous value.
        exog_feats (list): list of exogenous features for SARIMAX.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    train = df.filter(pl.col("date") < cutoff)
    eval = df.filter(pl.col("date") >= cutoff)

    train_pd = train.to_pandas()
    eval_pd = eval.to_pandas()

    chg_cols = [f"{label}_rolling_std_7"]

    cols_list = [label] + exog_feats + chg_cols

    train_pd = train_pd[cols_list]
    eval_pd = eval_pd[cols_list]

    return train_pd, eval_pd

def split_prophet_df(
        df: pd.DataFrame, cutoff: datetime
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """train/eval split for Prophet model.

    Args:
        df (pd.DataFrame): pandas dataframe set up for Prophet.
        cutoff (datetime): datetime object indicating the split date.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    df_train = df[df["ds"] < cutoff]
    df_eval = df[df["ds"] >= cutoff]

    return df_train, df_eval

Overwriting src/model_preprocess.py


leave this here for now  
trying to mess with `argparse` so it can run in the command line but i'll save  
that for last....

In [9]:
# import argparse

# ### parameters (use argparse module)

# # model hyperparameters
# DEFAULT_NUM_ESTIMATORS = 100
# DEFAULT_LEARNING_RATE = 0.01

# parser = argparse.ArgumentParser(
#     description="model hyperparameters"
# )

# parser.add_argument(
#     "-NUM_ESTIMATORS",
#     type=int,
#     default=DEFAULT_NUM_ESTIMATORS,
#     help="number of estimators"
# )
# parser.add_argument(
#     "-LEARNING_RATE",
#     type=float,
#     default=DEFAULT_LEARNING_RATE,
#     help="how fast the model learns"
# )

# # data parameters
# DEFAULT_LABEL = "move"

# parser.add_argument(
#     "-START_DATE",
#     type=str,
#     default=None,
#     help="data training start date"
# )

# parser.add_argument(
#     "-END_DATE",
#     type=str,
#     default=None,
#     help="data training end date"
# )

# parser.add_argument(
#     "-STOCKS",
#     type=list,
#     default=None,
#     help="stocks to forecast"
# )

# parser.add_argument(
#     "-LABEL",
#     type=str,
#     default=DEFAULT_LABEL,
#     help="one of 'move', 'open', 'close'; which of these values to forecast"
# )

# # cutoff value
# parser.add_argument(
#     "-CUTOFF",
#     type=datetime,
#     default=None,
#     help="cutoff value for train/test split"
# )

# # create args
# args = parser.parse_args()

# NUM_ESTIMATORS = args.NUM_ESTIMATORS
# LEARNING_RATE = args.LEARNING_RATE
# STOCKS = args.STOCKS
# START_DATE = args.START_DATE
# END_DATE = args.END_DATE
# LABEL = args.LABEL
# CUTOFF = args.CUTOFF

# NOW build the model

# training the model

## xgboost

In [5]:
%%writefile src/xgboost_model/train_xgboost_model.py
"""
contains a wrapper function for loading the data and training the XGBoost model.
forecasting is not done here, only model training.
"""
from __future__ import annotations

import polars as pl
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error
from datetime import datetime
from typing import Tuple, Dict, List, Optional

from src.load_data import load_stocks
from src.model_preprocess import train_test_split_cutoff
from src.xgboost_model.xgboost_etl import build_dataset


def _make_base_params(n_estimators: int, learning_rate: float) -> dict:
    return dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.005,
        objective="reg:squarederror"
    )


def _fit_point_model(X_train, y_train, params: dict) -> xgb.XGBRegressor:
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    return model 


def _fit_quantile_model(
    X_train, y_train, params: dict, alpha: float
) -> xgb.XGBRegressor:
    qparams = dict(params)
    qparams["objective"] = "reg:quantileerror" 
    qparams["quantile_alpha"] = alpha 
    model = xgb.XGBRegressor(**qparams)
    model.fit(X_train, y_train)
    return model 


def train_xgb_model(
    stocks: List[str],
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label_col: str,
    horizon: int,
    n_estimators: int,
    learning_rate: float,
    quantiles: Optional[List[float]] = None,
    label_mode: str = "log_return"
) -> Tuple[Dict[str, xgb.XGBRegressor], float, pl.DataFrame, List[str]]:
    """load raw data, preprocess, and train XGBoost model.

    trains either a point model (no quantiles) or a quantiles bundle (list of 
    floats).

    Args:
        stocks (list): single-item list of stock tickers.
        start_date (str): when to start the dataframe.
        end_date (str): final date of the dataframe.
        cutoff (datetime): cutoff datetime object for train/test splits.
        label_col (str): label column (y).
        horizon (int): forecast horizon.
        n_estimators (int): XGBoost `n_estimators` hyperparameter.
        learning_rate (float): XGBoost `learning_rate` hyperparameter.
        quantiles (Optional[List[float]]): list of quantiles to train models on.
        label_mode: whether the label is a log or a simple return.

    Raises:
        ValueError: cannot process more than one stock at a time.

    Returns:
        Tuple[Dict[str, xgb.XGBRegressor], float, pl.DataFrame, list]: XGBoost 
        regression model, RMSE value, full dataframe with features, features
        list.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")

    df_raw = load_stocks(
        stocks=stocks, start=start_date, end=end_date, use_polars=True
    )
    df_feat = build_dataset(
        df=df_raw, label_col=label_col, horizon=horizon, label_mode=label_mode
    )
    X_train, X_test, y_train, y_test = train_test_split_cutoff(
        df=df_feat, cutoff=cutoff, label_col="label"
    )
    feature_cols = X_train.to_pandas().columns.tolist()

    base_params = _make_base_params(
        n_estimators=n_estimators, learning_rate=learning_rate
    )

    models: Dict[str, xgb.XGBRegressor] = {}

    if not quantiles:
        m = _fit_point_model(X_train, y_train, base_params)
        preds = m.predict(X_test)
        rmse = root_mean_squared_error(y_test, preds)
        models["point"] = m 
        return models, rmse, df_feat, feature_cols
    
    for q in quantiles:
        key = f"q{int(round(q*100))}"
        try:
            models[key] = _fit_quantile_model(
                X_train, y_train, base_params, alpha=q
            )
        except TypeError as e:
            raise TypeError(
                "xgboost installed doesn't support sklearn quantile params "
                "('reg:quantileerror' / 'quantile_alpha'). "
                "upgrade xgboost or use the point model + residual bands."
            ) from e 
    
    if "q50" not in models:
        mid = sorted(quantiles)[len(quantiles)//2]
        mid_key = f"q{int(round(mid*100))}"
    else:
        mid_key = "q50" 
    
    preds = models[mid_key].predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)

    return models, rmse, df_feat, feature_cols
        

Overwriting src/xgboost_model/train_xgboost_model.py


# forecasting

this predicts future unseen values (not just test data)

!!!! will need to debug this to align the dates with what sarimax outputs

In [6]:
%%writefile src/xgboost_model/xgboost_forecaster.py
"""
contains a class for forecasting the future value from the trained XGBoost model.
"""
from __future__ import annotations 

import pandas as pd
import polars as pl
import xgboost as xgb
from datetime import timedelta, datetime
from typing import Dict, List 

from src.xgboost_model.xgboost_etl import prep_data_frame
from src.utils import build_trading_future_dates


class XGBExpiryForecaster:
    """
    forecast a horizon-ahead return distribution (or point) from the latest 
    features.
    """
    def __init__(
        self,
        models: Dict[str, xgb.XGBRegressor],
        feature_cols: List[str],
        label_mode: str = "log_return"
    ):
        self.models = models
        self.feature_cols = feature_cols
        self.label_mode = label_mode
    
    def _latest_feature_row(self, df_raw: pl.DataFrame) -> pd.DataFrame:
        df_feat = prep_data_frame(df_raw)
        return df_feat.select(self.feature_cols).tail(1).to_pandas()
    
    def _return_to_price(self, s0: float, r: float) -> float:
        if self.label_mode == "log_return":
            return float(s0 * (2.718281828459045 ** r))
        return float(s0 * (1.0 + r))
    
    def forecast_expiry(
        self,
        df_raw: pl.DataFrame,
        horizon: int,
        price_col: str = "close"
    ) -> pl.DataFrame:
        """forecast to expiry at +horizon trading days
        
        returns 1-row df with date + predicted return quantiles + price quantiles.

        Args:
            df_raw (pl.DataFrame): raw `yfinance` stock data.
            horizon (int): forecast horizon trading days.
            price_col (str, optional): the initial column off of which the label
                was built. defaults to 'close'. 
        
        Returns:
            pl.DataFrame: table with dates and predicted values.
        """
        last_date = df_raw["date"][-1]
        
        if isinstance(last_date, datetime):
            last_dt = last_date
        else:
            last_dt = datetime.combine(last_date, datetime.min.time())
        
        expiry_date = build_trading_future_dates(last_dt, horizon)[-1]

        s0 = float(df_raw[price_col][-1])
        X = self._latest_feature_row(df_raw)

        out = {"date": [expiry_date], "spot": [s0], "horizon": [horizon]}

        for name, m in self.models.items():
            rhat = float(m.predict(X)[0])
            out[f"pred_{name}_ret"] = [rhat]
            out[f"pred_{name}_px"] = [self._return_to_price(s0, rhat)]
        
        return pl.from_pandas(pd.DataFrame(out))


class XGBStockForecaster:
    """
    use the trained XGBoost model to make a forecast over a specified interval.
    """
    def __init__(
        self, model: xgb.XGBRegressor, feature_cols: list, label_col: str
    ):
        self.model = model
        self.feature_cols = feature_cols
        self.label_col = label_col
    
    def predict_one(self, df_raw: pl.DataFrame) -> float:
        """predict using the most recent row of features built from `df_raw`"""
        df_feat = prep_data_frame(df_raw)
        row_pd = df_feat.select(self.feature_cols).tail(1).to_pandas()
        pred = self.model.predict(row_pd)
        return float(pred[0])

Overwriting src/xgboost_model/xgboost_forecaster.py


# full modeling pipeline

raw data -> ETL -> split -> train a model -> evaluate model training -> forecast

In [7]:
%%writefile src/xgboost_model/xgboost_pipeline.py
"""
full pipeline for loading, transforming, training, and forecasting data for the
XGBoost model.
"""
from __future__ import annotations 

import pandas as pd
import polars as pl
from datetime import datetime
from typing import Tuple, Dict, List, Optional 

from src.load_data import load_stocks
from src.utils import build_forecast_dates
from src.xgboost_model.train_xgboost_model import train_xgb_model
from src.xgboost_model.xgboost_forecaster import XGBStockForecaster, XGBExpiryForecaster

def train_and_forecast_xgb_options(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label_col: str = "close",
    horizons: List[int] = [10, 21, 40],
    quantiles: Optional[List[float]] = [0.1, 0.5, 0.9],
    label_mode: str = "log_return",
    n_estimators: int = 400,
    learning_rate: float = 0.03,
) -> Tuple[pl.DataFrame, Dict[int, float]]:
    """full pipeline for forecasting options.

    for each horizon, train model(s) and forecast expiry distribution. returns
    combined forecast table (rows = horizons) and RMSEs per horizon.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        label_col (str, optional): which value to predict. defaults to "close".
        horizons (List[int], optional): horizon days to forecast. 
            defaults to [10, 21, 40].
        quantiles (Optional[List[float]], optional): quantile distributions to
            model on. defaults to [0.1, 0.5, 0.9].
        label_mode (str, optional): how to transform the label. defaults to
            "log_return".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 400.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter. 
            defaults to 0.03.

    Returns:
        Tuple[pl.DataFrame, Dict[int, float]]: dataframe of predicted values per
            horizon and the RMSE from training.
    """
    stocks = [ticker]
    df_raw = load_stocks(stocks, start_date, end_date, use_polars=True)

    forecasts = []
    rmses: Dict[int, float] = {}

    for h in horizons:
        models, rmse, df_feat, feature_cols = train_xgb_model(
            stocks=stocks,
            start_date=start_date,
            end_date=end_date,
            cutoff=cutoff,
            label_col=label_col,
            horizon=h,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            quantiles=quantiles,
            label_mode=label_mode
        )
        rmses[h] = rmse 

        forecaster = XGBExpiryForecaster(
            models=models, feature_cols=feature_cols, label_mode=label_mode
        )
        fc = forecaster.forecast_expiry(
            df_raw=df_raw, horizon=h, price_col=label_col
        )
        forecasts.append(fc)
    
    return pl.concat(forecasts, how="vertical"), rmses


def train_and_forecast_xgb(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    horizon_days: int,
    label_col: str = "close",
    n_estimators: int = 200,
    learning_rate: float = 0.05
) -> Tuple[pl.DataFrame, float]:
    """full pipeline for XGBoost model.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        label_col (str, optional): which value to predict. defaults to "close".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 200.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter.
            defaults to 0.05.

    Returns:
        Tuple[pl.DataFrame, float]: dataframe of predicted values per date and
            the RMSE from training.
    """
    stocks = [ticker]

    df_raw = load_stocks(stocks, start_date, end_date, use_polars=True)
    last_date = df_raw.select("date").max().item()
    future_dates = build_forecast_dates(last_date, horizon_days)

    preds = []
    rmses = []

    for h in range(1, horizon_days + 1):
        model, rmse, df_feat, feature_cols = train_xgb_model(
            stocks=stocks,
            start_date=start_date,
            end_date=end_date,
            cutoff=cutoff,
            label_col=label_col,
            horizon=h,
            n_estimators=n_estimators,
            learning_rate=learning_rate
        )

        forecaster = XGBStockForecaster(model, feature_cols, label_col=label_col)
        pred = forecaster.predict_one(df_raw)

        preds.append(pred)
        rmses.append(rmse)
    
    rmse_out = float(sum(rmses) / len(rmses))

    df_out = pl.from_pandas(
        pd.DataFrame({"date": future_dates, f"pred_{label_col}": preds})
    )

    return df_out, rmse_out

Overwriting src/xgboost_model/xgboost_pipeline.py


In [13]:
%%writefile src/sarimax_model/train_predict_sarimax_model.py
"""
full pipeline for loading the data, training, and forecasting with SARIMAX.
"""
import polars as pl
import pandas as pd
import pmdarima as pm 
from sklearn.metrics import root_mean_squared_error
from datetime import datetime, timedelta
from typing import Tuple

from src.load_data import load_stocks
from src.sarimax_model.sarimax_etl import *
from src.model_preprocess import split_ar_on_cutoff
from src.utils import build_forecast_dates


def fit_sarimax(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int,
    eval_mode: bool = True
):
    """fit the actual SARIMAX model.

    has two modes: eval and forecast. eval mode uses the train-eval split to 
    gauge how accurate the forecasts are (using RMSE). forecast mode uses the
    full dataset to train and make a forecast.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        eval_mode (bool, optional): whether to run the model as an evaluation of
            performance or as a full forecast. defaults to True (i.e., evaluate 
            the model performance).

    Returns:
        _type_: output depends on `eval_mode`. returns a float (RMSE) if 
            `eval_mode` is True, or a table (date, predicted value) if 
            `eval_mode` is False.
    """
    stock = [ticker]

    df_raw = load_stocks(stock, start_date, end_date)
    df_idx = build_df_with_indices(df_raw, label, start_date, end_date)

    feats = [
        "date",
        "dow",
        "month",
        "mon_or_fri",
        "volume",
        "SPY_close",
        "SPY_volume",
        "QQQ_close",
        "QQQ_volume",
        "IWM_close",
        "IWM_volume",
        "VXX_close",
        "VXX_volume",
        "UUP_close",
        "UUP_volume",
        "HYG_close",
        "HYG_volume",
        "LQD_close",
        "LQD_volume"
    ]

    exog_cols = [feat for feat in feats if feat != "date"]

    _sarima_hyperparams = {
        "start_p": 1,
        "start_q": 1,
        "test": "adf",
        "max_p": 3,
        "max_q": 3,
        "m": 5,
        "start_P": 0,
        "seasonal": True,
        "d": None,
        "D": None,
        "trace": False,
        "error_action": "ignore",
        "suppress_warnings": True,
        "stepwise": True
    }
    
    if eval_mode:
        df_train, df_eval = split_ar_on_cutoff(df_idx, cutoff, "close", feats)

        sarimax_model = pm.auto_arima(
            df_train[[label]],
            exogenous=df_train[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=len(df_eval),
            return_conf_int=True,
            exogenous=df_eval[exog_cols]
        )

        fitted = pd.DataFrame(fitted, columns=["pred"]).reset_index(drop=True)
        df_eval["pred"] = fitted["pred"]
        rmse = root_mean_squared_error(df_eval[["close"]], df_eval[["pred"]])
        
        return rmse
    else:
        df = df_idx.to_pandas()

        sarimax_model = pm.auto_arima(
            df[[label]],
            exogenous=df[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=horizon_days,
            return_conf_int=True,
            exogenous=df[exog_cols]
        )
        fitted = pd.DataFrame(fitted, columns=[f"pred_{label}"]).reset_index(
            drop=True
        )
        ci_series = pd.DataFrame(
            confint, columns=["lower_bound", "upper_bound"]
        )
        
        df_out = build_forecast_dates(
            end_date, horizon_days, skip_weekends=True
        )
        
        df_out[f"pred_{label}"] = fitted[f"pred_{label}"]
        df_out["lower_bound"] = ci_series["lower_bound"]
        df_out["upper_bound"] = ci_series["upper_bound"]

        return df_out.sort_values(by="date", ascending=True)

def sarimax_wrapper(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int
) -> Tuple[pl.DataFrame, float]:
    """wrapper to run both versions of `fit_sarimax`.

    gets training results (`eval_mode == True`) and forecast results (`eval_mode
    == False`).

    Args:
        ticker (str): stock ticker to predict
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.

    Returns:
        Tuple[pl.DataFrame, float]: table of predictions (date, predicted value)
            and the RMSE from training.
    """
    rmse = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=True
    )

    forecasts = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=False
    )

    return pl.from_pandas(forecasts), rmse

Overwriting src/sarimax_model/train_predict_sarimax_model.py


In [14]:
%%writefile src/prophet_model/prophet_model_pipeline.py
"""_summary_

Returns:
    _type_: _description_
"""
import pandas as pd 
import polars as pl 
from prophet import Prophet 
from datetime import datetime
from typing import Tuple
from sklearn.metrics import root_mean_squared_error

import warnings 
warnings.filterwarnings("ignore")

from src.prophet_model.prophet_etl import build_prophet_df
from src.utils import build_forecast_dates
from src.model_preprocess import split_prophet_df


class ProphetForecaster:
    """wrapper class to train and forecast Prophet model."""

    def __init__(self, base_params: dict | None = None):
        self.base_params = base_params or {"interval_width": 0.95}
        self.model = None
        self.reg_cols = [] 
    
    def train_model(
        self,
        ticker: str,
        start_date: str,
        end_date: str,
        cutoff: datetime,
        label: str = "close",
        indexes: list[str] | None = None
    ) -> Tuple[float, pd.DataFrame]:
        """build data, train and evaluate Prophet model.

        data includes the stock label (defaults to close), index fund values,
        returns, RSI (7, 14, 21 days), and lag features of label (1, 7, 30 days)
        to predict future label values.

        Args:
            stock (list): stock to be predicted.
            start_date (str): start of historical price data.
            end_date (str): end of historical price data.
            cutoff (datetime): date to split for train/eval.
            label (str, optional): value to predict. defaults to "close".
            indexes (list[str] | None, optional): specific index funds to add to
                feature space. defaults to None.

        Returns:
            Tuple[float, pd.DataFrame]: RMSE value for evaluation and eval
                dataframe.
        """
        stock = [ticker]
        
        df = build_prophet_df(stock, start_date, end_date, label, indexes)
        df_train, df_eval = split_prophet_df(df, cutoff)

        self.reg_cols = [c for c in df.columns if c not in ["ds", "y"]]

        pr_model = Prophet(**self.base_params)
        for col in self.reg_cols:
            pr_model.add_regressor(col)

        pr_model.fit(df_train[["ds", "y"] + self.reg_cols])
        self.model = pr_model

        df_eval_future = df_eval[["ds"] + self.reg_cols].copy()
        forecast_eval = self.model.predict(df_eval_future)

        df_forecast = forecast_eval[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        df_eval_out = df_eval[["ds", "y"]]
        df_eval_out = df_eval_out.merge(df_forecast, on="ds", how="inner")

        rmse = root_mean_squared_error(df_eval["y"], df_eval_out["yhat"])
        
        return rmse, df_eval_out
    
    def _build_future_regressors(
            self, df_full: pd.DataFrame, future_dates: pd.DataFrame
        ) -> pd.DataFrame:
        """stand-in function to make a dataframe for future predictions.

        will need to be added to if this were to ever go live for the sake of
        adding future regressors (and not just dates).

        Args:
            df_full (pd.DataFrame): full features dataframe.
            future_dates (pd.DataFrame): dataframe of days in the future.

        Returns:
            pd.DataFrame: single row containing date and regressors for making
                the forecast.
        """
        last_row = df_full.sort_values("ds").iloc[-1]
        date_list = future_dates["date"].tolist()
        
        rows = []

        for d in date_list:
            row = {"ds": d}
            for col in self.reg_cols:
                row[col] = last_row[col]
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def make_prediction(
        self,
        df_full: pd.DataFrame,
        horizon_days: int,
        label: str = "close"
    ) -> pl.DataFrame:
        """generate predictions `horizon_days` in the future.

        returns a polars dataframe for in-line displays.

        Args:
            df_full (pd.DataFrame): full features dataframe.
            horizon_days (int): number of days to forecast into the future.
            label (str, optional): specific value to forecast. defaults to 
            "close".

        Returns:
            pl.DataFrame: polars dataframe containing future dates, predicted
                values, and upper/lower bound CIs (95%).
        """
        assert self.model is not None

        last_date = df_full["ds"].max()

        future_dates = build_forecast_dates(
            last_date,
            horizon_days=horizon_days,
            skip_weekends=True
        )

        future_regs = self._build_future_regressors(df_full, future_dates)

        future_df = future_regs[["ds"] + self.reg_cols]

        forecast = self.model.predict(future_df)
        out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        df_out = pl.from_pandas(out)

        return df_out.select(
            pl.col("ds").alias("date"),
            pl.col("yhat").alias(f"pred_{label}"),
            pl.col("yhat_lower").alias("lower_bound"),
            pl.col("yhat_upper").alias("upper_bound")
        )

Overwriting src/prophet_model/prophet_model_pipeline.py


In [15]:
%%writefile src/utils.py
"""
utility functions
    build_forecast_dates
"""
from __future__ import annotations

from datetime import timedelta, date, datetime
import pandas as pd
from typing import List, Union

def build_forecast_dates(last_date, horizon_days: int) -> pd.DatetimeIndex:
    """return the next `horizon_days` trading dates (mon-fri), starting AFTER `last_date`.

    Args:
        last_date (_type_): final date of the training window.
        horizon_days (int): number of days for the forecast to project.

    Returns:
        pd.DatetimeIndex: index of datetimes for the forecast.
    """
    if isinstance(last_date, str):
        last_date = pd.to_datetime(last_date)
    
    dates = []
    d = pd.to_datetime(last_date)

    while len(dates) < horizon_days:
        d = d + pd.Timedelta(days=1)
        if d.weekday() < 5:
            dates.append(d)
    
    return pd.DatetimeIndex(dates)


def next_trading_day(d: datetime) -> datetime:
    """advance to next weekday (mon-fri)"""
    d = d + timedelta(days=1)
    while d.weekday() >= 5:
        d = d + pd.Timedelta(days=1)
    return d


def build_trading_future_dates(
    last_date: datetime, n_trading_days: int
) -> List[datetime]:
    """return next N trading dates strictly after `last_date`"""
    out = []
    d = last_date
    for _ in range(n_trading_days):
        d = next_trading_day(d)
        out.append(d)
    return out

Overwriting src/utils.py


In [3]:
# %%writefile workflows/train_and_forecast_model.py
"""
script for getting all model results.
"""
import pandas as pd 
import polars as pl 
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from timeit import default_timer as timer

import warnings
warnings.filterwarnings("ignore")

from src.xgboost_model.xgboost_pipeline import train_and_forecast_xgb_options

ticker = "FUBO"
horizons = [10, 21, 40, 90, 180]

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - relativedelta(years=10)).strftime("%Y-%m-%d")

max_horizon = max(horizons)
if max_horizon > 90:
    cm = 12
elif max_horizon > 40:
    cm = 6
else:
    cm = 2

print(
    f"\n\nmax horizon is {max_horizon} days, "
    f"setting cutoff to {cm} months for train-test split."
)

cutoff = (datetime.today() - relativedelta(months=cm))
print("")
label = "close"

print(
    f"forecasting '{ticker}' \033[4m{label}\033[0m prices "
    f"over the next {horizons} days"
)
print(
    f"training models from '{start_date}' to '{end_date}', "
    f"splitting on '{cutoff.strftime("%Y-%m-%d")}'\n\n"
)

timer_xgb_start = timer()

xgb_forecasts, xgb_rmse = train_and_forecast_xgb_options(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    cutoff=cutoff,
    label_col=label,
    horizons=horizons,
    quantiles=[0.1, 0.5, 0.9],
    label_mode="log_return",
    n_estimators=400,
    learning_rate=0.03
)
timer_xgb_end = timer() - timer_xgb_start
print(f"XGBoost run duration: {timer_xgb_end:.5f} seconds\n\n")

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=1000):
    print(xgb_forecasts)
    print(xgb_rmse)



max horizon is 180 days, setting cutoff to 12 months for train-test split.

forecasting 'FUBO' close prices over the next [10, 21, 40, 90, 180] days
training models from '2016-02-21' to '2026-02-21', splitting on '2025-02-21'




[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


XGBoost run duration: 11.48584 seconds


shape: (5, 9)
┌───────────┬──────┬─────────┬───────────┬───────────┬───────────┬───────────┬──────────┬──────────┐
│ date      ┆ spot ┆ horizon ┆ pred_q10_ ┆ pred_q10_ ┆ pred_q50_ ┆ pred_q50_ ┆ pred_q90 ┆ pred_q90 │
│ ---       ┆ ---  ┆ ---     ┆ ret       ┆ px        ┆ ret       ┆ px        ┆ _ret     ┆ _px      │
│ datetime[ ┆ f64  ┆ i64     ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---      ┆ ---      │
│ ns]       ┆      ┆         ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64      ┆ f64      │
╞═══════════╪══════╪═════════╪═══════════╪═══════════╪═══════════╪═══════════╪══════════╪══════════╡
│ 2026-03-0 ┆ 1.24 ┆ 10      ┆ -0.093258 ┆ 1.129589  ┆ 0.046171  ┆ 1.298594  ┆ 0.088339 ┆ 1.354524 │
│ 6         ┆      ┆         ┆           ┆           ┆           ┆           ┆          ┆          │
│ 00:00:00  ┆      ┆         ┆           ┆           ┆           ┆           ┆          ┆          │
│ 2026-03-2 ┆ 1.24 ┆ 21      ┆ -0.05

In [ ]:
quantiles = [0.10, 0.50, 0.90]


In [ ]:
## save for later
# from src.sarimax_model.train_predict_sarimax_model import *
# from src.prophet_model.prophet_model_pipeline import ProphetForecaster
# from src.prophet_model.prophet_etl import build_prophet_df



# timer_smax_start = timer()

# smax_forecasts, smax_rmse = sarimax_wrapper(
#     ticker=ticker,
#     start_date=start_date,
#     end_date=end_date,
#     label=label,
#     cutoff=cutoff,
#     horizon_days=horizon,    
# )
# timer_smax_end = timer() - timer_smax_start

# print(f"\nSARIMAX test RMSE on holdout: {smax_rmse:.4f}\n")
# print(f"SARIMAX run duration: {timer_smax_end:.5f} seconds\n\n")
# print(smax_forecasts)
# print("\n\n")


# timer_prph_start = timer()

# idxs = ["VXX", "QQQ", "SPY", "IWM"]

# forecaster = ProphetForecaster()
# prph_rmse, eval_df = forecaster.train_model(
#     ticker=ticker,
#     start_date=start_date,
#     end_date=end_date,
#     cutoff=cutoff,
#     label=label,
#     indexes=idxs
# )
# df_full = build_prophet_df(
#     stocks=[ticker], start=start_date, end=end_date, label=label, indexes=idxs
# )
# prph_forecasts = forecaster.make_prediction(
#     df_full=df_full, horizon_days=horizon
# )
# timer_prph_end = timer() - timer_prph_start

# print(f"\nProphet test RMSE on holdout: {prph_rmse:.4f}\n")
# print(f"Prophet run duration: {timer_prph_end:.5f} seconds\n\n")
# print(prph_forecasts)
# print("\n\n")